# Retrain CatBoost and XGBoost on Train + Validation

## Natural-language request

> Retrain CatBoost and XGBoost on the training and validation data both, in another notebook, and then do the final test run.

This notebook refits the preprocessor and the two models on **train + val** (arrivals 1 July 2015 to 31 December 2016), rechecks for leakage, and **freezes run 3's settings before test is opened**. The test run itself is notebook 16.

## What the contract allows, and what stays fixed

`docs/EVALUATION_CONTRACT.md`: *"You may refit on train and choose together before the proof run, as long as you keep every setting you picked."* So:

| Kept exactly | Value |
|---|---|
| Inputs | The eight inputs from notebook 06 (four provisional) |
| Preprocessing | Same steps as notebook 10: one-hot with categories under **30** bookings grouped, `log(1 + count)` and scaling for the two counts |
| Model settings | Same as notebook 11 (CatBoost: 300 rounds, depth 4, rate 0.05; XGBoost: 300 rounds, depth 4, rate 0.05, 80% rows/columns), same seed |
| Oversell rule and buffers | k = max(0, ⌊expected⌋ − buffer); per hotel **City 6, Resort 3** for both models; shared **5** |
| Flat rule | **Train** hotel rates (as the contract specifies), buffers City 20 / Resort 4; shared 16 |
| Primary method | **CatBoost** (chosen on val, notebook 13); XGBoost reported alongside |

## Disclosure

Run 1 (CatBoost fitted on train only) has already been scored on test (notebook 14: EUR 150,459 more than the flat rule). This refit is decided **after** seeing that result, so it is a post-proof change: **every run is reported** (`docs/DECISIONS.md`).

**One risk to state before the run.** The buffers were chosen on val for the **train-only** models. Models refitted with val's data may expect more cancellations: val has far fewer bookings with prior cancellations and a higher cancellation rate among everyone else. A higher expected count with the **same** buffer means a larger oversell, and so more walks. The contract does not allow the buffers to be re-chosen on test, and there is no unseen data left to re-choose them on. **Step 6 measures how far the expected counts moved**, so the risk is visible before test is opened.

**Test is never opened in this notebook.**

## Setup

In [1]:
from pathlib import Path
import datetime
import hashlib
import json
import sys
import warnings

import joblib
import numpy as np
import pandas as pd
from IPython.display import display
from catboost import CatBoostClassifier
from sklearn.compose import ColumnTransformer
from sklearn.metrics import log_loss, roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler
from xgboost import XGBClassifier

# Find the repository so this works from the repo folder or notebooks/.
repo_root = None
for folder in (Path.cwd(), *Path.cwd().parents):
    if (folder / "src" / "contract.py").exists():
        repo_root = folder
        break
if repo_root is None:
    raise FileNotFoundError("Could not find the project src/ folder.")
if str(repo_root / "src") not in sys.path:
    sys.path.insert(0, str(repo_root / "src"))

import contract

split_dir = repo_root / "data" / "splits"
manifest = json.loads((split_dir / "split_columns.json").read_text(encoding="utf-8"))
inputs, provisional, label = manifest["inputs"], manifest["provisional_inputs"], manifest["label"]
run1_freeze = json.loads((repo_root / "models" / "frozen_choice.json").read_text(encoding="utf-8"))
nb10_settings = json.loads((repo_root / "data" / "preprocessed" / "feature_names.json").read_text(encoding="utf-8"))


def load_split(name):
    # Test stays closed: it is opened only in notebook 16, after the freeze below.
    if name not in ("train", "val"):
        raise ValueError("This notebook only opens train and val.")
    return pd.read_csv(split_dir / f"{name}.csv", index_col="row",
                       parse_dates=["arrival_date"], dtype={"agent": str, "company": str})


train, val = load_split("train"), load_split("val")
train_val = pd.concat([train, val])
X_fit, y_fit = train_val[inputs], train_val[label]

checks = []


def passed(leakage, what):
    checks.append({"leakage": leakage, "check": what, "result": "PASS"})
    print(f"PASS  [{leakage}] {what}")


print(f"train {len(train):,} + val {len(val):,} = {len(train_val):,} bookings; "
      f"arrivals {train_val['arrival_date'].min():%Y-%m-%d} to {train_val['arrival_date'].max():%Y-%m-%d}")

train 49,157 + val 29,546 = 78,703 bookings; arrivals 2015-07-01 to 2016-12-31


## Step 1: check the combined data

Train + val must hold every train and val booking exactly once and **nothing from the test period**. The inputs must be the same eight columns as before.

In [2]:
assert train_val.index.is_unique and len(train_val) == manifest["periods"]["train"]["rows"] + manifest["periods"]["val"]["rows"]
passed("L3", "train + val holds every train and val booking exactly once")
assert train_val["arrival_date"].max() < pd.Timestamp(contract.PROOF_START)
passed("L7", f"no booking arrives on or after {contract.PROOF_START} (test period)")
outcome_and_scoring = {"is_canceled", "reservation_status", "reservation_status_date", *manifest["scoring_only"]}
assert list(X_fit.columns) == inputs and not set(inputs) & outcome_and_scoring
passed("L1", "inputs are the same eight columns; no label, outcome or scoring-only column")

PASS  [L3] train + val holds every train and val booking exactly once
PASS  [L7] no booking arrives on or after 2017-01-01 (test period)
PASS  [L1] inputs are the same eight columns; no label, outcome or scoring-only column


## Step 2: refit the preprocessor on train + val

The same construction as notebook 10, refitted on train + val. Category lists, the rare categories and the scaling statistics now come from both periods. The checks from notebook 10 are repeated against train + val.

In [3]:
MIN_BOOKINGS = nb10_settings["min_bookings_per_category"]
category_inputs = ["hotel", "market_segment", "distribution_channel", "agent", "company"]
count_inputs = ["previous_cancellations", "previous_bookings_not_canceled"]
flag_inputs = ["is_repeated_guest"]
assert sorted(category_inputs + count_inputs + flag_inputs) == sorted(inputs)
warnings.filterwarnings("ignore", message="Found unknown categories")


def make_preprocessor():
    return ColumnTransformer(
        transformers=[
            ("category", OneHotEncoder(min_frequency=MIN_BOOKINGS, handle_unknown="infrequent_if_exist",
                                       drop="if_binary", sparse_output=False), category_inputs),
            ("count", Pipeline([("log1p", FunctionTransformer(np.log1p, feature_names_out="one-to-one")),
                                ("scale", StandardScaler())]), count_inputs),
            ("flag", "passthrough", flag_inputs),
        ],
        remainder="drop",
    ).set_output(transform="pandas")


preprocessor = make_preprocessor().fit(X_fit)
X_prepared = preprocessor.transform(X_fit)

# L3: everything learnt equals what is counted in train + val alone.
encoder = preprocessor.named_transformers_["category"]
for column, known, rare in zip(category_inputs, encoder.categories_, encoder.infrequent_categories_):
    counts = X_fit[column].value_counts()
    assert set(known) == set(counts.index)
    assert set(rare if rare is not None else []) == set(counts[counts < MIN_BOOKINGS].index)
scaler = preprocessor.named_transformers_["count"].named_steps["scale"]
logged = np.log1p(X_fit[count_inputs])
assert np.allclose(scaler.mean_, logged.mean()) and np.allclose(scaler.scale_, logged.std(ddof=0))
passed("L3", "categories, rare categories and scaling statistics equal those counted in train + val")

# L4: the label plays no part.
shuffled = pd.Series(np.random.default_rng(0).permutation(y_fit.to_numpy()), index=y_fit.index)
assert make_preprocessor().fit(X_fit, shuffled).transform(X_fit).equals(X_prepared)
passed("L4", "fitting with a shuffled label gives identical output")

# L5: each row is transformed on its own.
bounds = np.linspace(0, len(X_fit), 6, dtype=int)
pieces = pd.concat([preprocessor.transform(X_fit.iloc[a:b]) for a, b in zip(bounds[:-1], bounds[1:])])
assert pieces.equals(X_prepared)
passed("L5", "output is the same whole or in pieces")

# L6: only the eight inputs reach the output.
names = list(preprocessor.get_feature_names_out())
assert all(any(n.split("__", 1)[1] == c or n.split("__", 1)[1].startswith(c + "_") for c in inputs) for n in names)
assert np.isfinite(X_prepared.to_numpy(dtype=float)).all()
passed("L6", f"all {len(names)} output columns come from the eight inputs; no missing or infinite values")

print(f"\n{len(names)} prepared columns (notebook 10 had {len(nb10_settings['output_columns'])}): "
      "the column count can change because categories reaching 30 bookings differ once val is added.")

PASS  [L3] categories, rare categories and scaling statistics equal those counted in train + val


PASS  [L4] fitting with a shuffled label gives identical output


PASS  [L5] output is the same whole or in pieces


PASS  [L6] all 167 output columns come from the eight inputs; no missing or infinite values

167 prepared columns (notebook 10 had 143): the column count can change because categories reaching 30 bookings differ once val is added.


## Step 3: refit CatBoost and XGBoost on train + val

Exactly the settings of notebook 11, with no class weights. Nothing is tuned.

In [4]:
SEED = 0
models = {
    "CatBoost": CatBoostClassifier(iterations=300, depth=4, learning_rate=0.05, random_seed=SEED, verbose=0,
                                   allow_writing_files=False),
    "XGBoost": XGBClassifier(n_estimators=300, max_depth=4, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8,
                             eval_metric="logloss", random_state=SEED, n_jobs=-1),
}
X = X_prepared.to_numpy(dtype=float)
for name, model in models.items():
    model.fit(X, y_fit.to_numpy())
    print(f"Fitted {name} on {len(X):,} train + val bookings")

# The settings match the train-only models of notebook 11.
model_dir = repo_root / "data" / "models"
for name, model in models.items():
    old = joblib.load(model_dir / f"{name}.joblib").get_params()
    new = model.get_params()
    differing = {k for k in new if k in old and str(new[k]) != str(old[k])}
    assert not differing, f"{name}: settings differ from notebook 11: {differing}"
passed("settings", "CatBoost and XGBoost settings are identical to notebook 11")

Fitted CatBoost on 78,703 train + val bookings


Fitted XGBoost on 78,703 train + val bookings
PASS  [settings] CatBoost and XGBoost settings are identical to notebook 11


## Step 4: in-sample fit, by period

These are **in-sample** numbers (the models learnt from these bookings), shown only to confirm the fits are sensible and to see whether the refit now matches val's cancellation level. They are not performance figures.

In [5]:
probabilities = {name: pd.Series(m.predict_proba(X)[:, 1], index=train_val.index) for name, m in models.items()}
rows = []
for name, p in probabilities.items():
    for period, part in {"train": train, "val": val}.items():
        y, q = part[label], p.loc[part.index]
        rows.append({"model": name, "period": period, "log loss": log_loss(y, q), "ROC AUC": roc_auc_score(y, q),
                     "mean predicted rate": q.mean(), "actual rate": y.mean()})
display(pd.DataFrame(rows).set_index(["model", "period"]).style.format(
    {"log loss": "{:.4f}", "ROC AUC": "{:.3f}", "mean predicted rate": "{:.1%}", "actual rate": "{:.1%}"}))

## Step 5: how far did the expected counts move?

The frozen buffers were chosen for the **train-only** models' expected counts. This compares, on the val nights, the expected cancellations per hotel-night from the train-only models (notebook 12) with those from the refitted models, **with no outcome used**: it is a difference between two forecasts.

If the refitted models expect, say, 4 more cancellations per City Hotel night, the same 6-room buffer now oversells about 4 rooms more. That is a direct walk risk on test. It is measured here, before test is opened, and reported, not corrected: the buffers stay frozen.

In [6]:
old_probabilities = pd.read_csv(model_dir / "val_probabilities.csv", index_col="row").loc[val.index]
rows = []
for name in models:
    old = contract.nightly_table(val[contract.REQUIRED], old_probabilities[name])["expected"]
    new = contract.nightly_table(val[contract.REQUIRED], probabilities[name].loc[val.index])["expected"]
    shift = new - old
    for hotel in ["group", "City Hotel", "Resort Hotel"]:
        s = shift if hotel == "group" else shift.xs(hotel, level="hotel")
        rows.append({"model": name, "hotel": hotel, "mean change in expected cancellations per night (rooms)": s.mean(),
                     "largest increase (rooms)": s.max()})
shift_table = pd.DataFrame(rows).set_index(["model", "hotel"])
display(shift_table.style.format("{:+.2f}"))

## Step 6: save the refitted models and freeze run 3

Saves to `data/models_train_val/` and writes `models/frozen_choice_run3_train_val.json` with fingerprints of the saved files. Notebook 16 refuses to run if any file differs from its fingerprint.

In [7]:
def sha256(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()


out_dir = repo_root / "data" / "models_train_val"
out_dir.mkdir(parents=True, exist_ok=True)
joblib.dump(preprocessor, out_dir / "preprocessor.joblib")
for name, model in models.items():
    joblib.dump(model, out_dir / f"{name}.joblib")
(out_dir / "feature_names.json").write_text(json.dumps({"inputs": inputs, "output_columns": names}, indent=2) + "\n",
                                            encoding="utf-8")

# Reload and confirm identical predictions.
reloaded_pre = joblib.load(out_dir / "preprocessor.joblib")
X_check = reloaded_pre.transform(X_fit).to_numpy(dtype=float)
for name in models:
    assert np.allclose(joblib.load(out_dir / f"{name}.joblib").predict_proba(X_check)[:, 1], probabilities[name].to_numpy())
passed("save", "reloaded preprocessor and models reproduce the fitted predictions")

rel = lambda p: str(p.relative_to(repo_root)).replace("\\", "/")
freeze = {
    "frozen_at": datetime.datetime.now().isoformat(timespec="seconds"),
    "run": 3,
    "frozen_before_test": True,
    "changed_after_test": True,
    "disclosure": ("Run 3 refits the preprocessor, CatBoost and XGBoost on train + choose, decided after run 1's test result "
                   "(CatBoost train-only: EUR 150,459 more than the flat rule). All settings picked on choose are kept. "
                   "Run 2 (XGBoost train-only) was frozen but never executed. All runs are reported."),
    "fitted_on": "train + choose (arrivals 2015-07-01 to 2016-12-31)",
    "primary_method": "CatBoost",
    "also_reported": ["XGBoost"],
    "preprocessor_file": rel(out_dir / "preprocessor.joblib"),
    "preprocessor_sha256": sha256(out_dir / "preprocessor.joblib"),
    "models": {name: {"file": rel(out_dir / f"{name}.joblib"), "sha256": sha256(out_dir / f"{name}.joblib")} for name in models},
    "rule": run1_freeze["rule"],
    "primary": {"buffer_structure": "separate buffer per hotel; group-wide walk ceiling only",
                "model_buffers": {"CatBoost": {"City Hotel": 6, "Resort Hotel": 3}, "XGBoost": {"City Hotel": 6, "Resort Hotel": 3}},
                "flat_rule_buffers": run1_freeze["primary"]["flat_rule_buffers"]},
    "secondary_reported_for_transparency": {"buffer_structure": "one shared buffer for both hotels",
                                            "model_buffers": {"CatBoost": 5, "XGBoost": 5},
                                            "flat_rule_buffer": run1_freeze["secondary_reported_for_transparency"]["flat_rule_buffer"]},
    "flat_rule_hotel_rates_from_train": run1_freeze["flat_rule_hotel_rates_from_train"],
    "fairness_check_flat_rule_rates_from_train_and_choose": train_val.groupby("hotel")[label].mean().round(6).to_dict(),
    "walk_ceiling_per_1000_arrivals": contract.WALK_CEILING,
    "walk_cost_eur": contract.WALK_COST,
    "val_expected_count_shift_rooms": {f"{m} / {h}": round(float(v), 2) for (m, h), v in
                                       shift_table.iloc[:, 0].items()},
}
freeze_path = repo_root / "models" / "frozen_choice_run3_train_val.json"
freeze_path.write_text(json.dumps(freeze, indent=2) + "\n", encoding="utf-8")
print(f"Froze run 3 in {rel(freeze_path)} at {freeze['frozen_at']}. Test has not been opened.")
display(pd.DataFrame(checks))

PASS  [save] reloaded preprocessor and models reproduce the fitted predictions
Froze run 3 in models/frozen_choice_run3_train_val.json at 2026-11-18T02:05:41. Test has not been opened.


,leakage,check,result
0,L3,train + val holds every train and val booking ...,PASS
1,L7,no booking arrives on or after 2017-01-01 (tes...,PASS
2,L1,"inputs are the same eight columns; no label, o...",PASS
3,L3,"categories, rare categories and scaling statis...",PASS
4,L4,fitting with a shuffled label gives identical ...,PASS
5,L5,output is the same whole or in pieces,PASS
6,L6,all 167 output columns come from the eight inp...,PASS
7,settings,CatBoost and XGBoost settings are identical to...,PASS
8,save,reloaded preprocessor and models reproduce the...,PASS


## Summary

- **CatBoost and XGBoost were refitted on train + val** with the preprocessing and model settings unchanged. The preprocessing leakage checks were repeated against train + val, and test was not opened.
- **Run 3 is frozen** in `models/frozen_choice_run3_train_val.json`: CatBoost primary, XGBoost alongside, val-chosen buffers kept, flat rule with train rates. A flat rule with train + val rates is reported as a fairness check.
- **Step 5 shows how far the expected counts moved** under the frozen buffers, which is the walk risk to watch on test.

**Next:** notebook 16, the final test run.